# Trustpilot Review Scraper – FirstPort

This notebook scrapes **all public reviews** for [firstport.co.uk on Trustpilot](https://www.trustpilot.com/review/www.firstport.co.uk)  
using the open‑source [`trustpilot_scraper`](https://github.com/irfanalidv/trustpilot_scraper) library and saves them to CSV.

> **Notes**  
> • Scraping respects Trustpilot’s rate‑limits (2 s delay/page).  
> • For anything beyond internal analysis, review Trustpilot’s current ToS.  
> • The code should run as‑is in any recent Jupyter environment (Python 3.8+).

# Setup

In [ ]:
# Install required packages
!pip install -U -q requests beautifulsoup4 pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 99.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd
from tqdm.notebook import tqdm  # Use tqdm.notebook for Jupyter compatibility
import re
import os
from datetime import datetime

def get_reviews_from_page(url):
    """Get reviews from a specific Trustpilot page URL"""
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5"
        }

        req = requests.get(url, headers=headers)
        req.raise_for_status()

        soup = BeautifulSoup(req.text, 'html.parser')

        # Try to find total review count for validation
        total_reviews_pattern = re.compile(r'showing \d+-\d+ of ([\d,]+) reviews', re.IGNORECASE)
        pagination_text = soup.find(string=total_reviews_pattern)
        if pagination_text:
            match = total_reviews_pattern.search(pagination_text)
            if match:
                total_count = match.group(1).replace(',', '')
                print(f"Site reports approximately {total_count} total reviews")

        reviews_raw = soup.find("script", id="__NEXT_DATA__")
        if not reviews_raw:
            print(f"WARNING: Could not find __NEXT_DATA__ script on page {url}")
            return []

        reviews_raw = json.loads(reviews_raw.string)

        # Check if we have the expected structure
        if not (reviews_raw.get("props") and
                reviews_raw["props"].get("pageProps") and
                reviews_raw["props"]["pageProps"].get("reviews")):
            print(f"WARNING: Unexpected data structure on page {url}")
            return []

        reviews = reviews_raw["props"]["pageProps"]["reviews"]

        # Add page info to each review for tracking
        page_num = int(url.split("page=")[-1]) if "page=" in url else 1
        for review in reviews:
            review['_page_number'] = page_num

        # Look for pagination info
        pagination = reviews_raw["props"]["pageProps"].get("pagination", {})
        if pagination:
            current = pagination.get("page", "?")
            total = pagination.get("totalPages", "?")
            print(f"Page {current}/{total}")

        return reviews
    except Exception as e:
        print(f"Error fetching page {url}: {e}")
        return []

def scrape_trustpilot_reviews(base_url, start_page=1, end_page=None, max_pages=None, verbose=True):
    """
    Scrape Trustpilot reviews with page range control

    Parameters:
    - base_url: The base URL of the Trustpilot page
    - start_page: Page number to start scraping from (default: 1)
    - end_page: Page number to end scraping at (inclusive, default: None = no limit)
    - max_pages: Maximum number of pages to scrape (default: None = no limit)
    - verbose: Whether to print detailed progress information

    Returns:
    - List of review dictionaries
    """
    reviews_data = []
    page_number = start_page
    total_reviews = 0
    retry_count = 0
    max_retries = 3
    empty_pages = 0
    max_empty_pages = 3  # Stop after this many consecutive empty pages
    pages_scraped = 0

    # Calculate effective end_page if max_pages is specified
    if max_pages is not None:
        effective_end_page = start_page + max_pages - 1
        if end_page is not None:
            effective_end_page = min(effective_end_page, end_page)
    else:
        effective_end_page = end_page

    if verbose:
        if effective_end_page:
            print(f"Scraping pages {start_page} to {effective_end_page}")
        else:
            print(f"Scraping from page {start_page} onwards")

    # Create progress bar - using tqdm.notebook for Jupyter compatibility
    pbar = tqdm(desc="Scraping pages", unit="page")

    try:
        while True:
            url = f"{base_url}?page={page_number}"
            if verbose:
                pbar.set_description(f"Page {page_number}")

            reviews = get_reviews_from_page(url)

            if not reviews:
                empty_pages += 1
                if verbose:
                    print(f"No reviews found on page {page_number} (Empty pages: {empty_pages}/{max_empty_pages})")

                if empty_pages >= max_empty_pages:
                    if verbose:
                        print(f"Stopping after {max_empty_pages} consecutive empty pages")
                    break

                # Try next page anyway
                page_number += 1
                pages_scraped += 1
                pbar.update(1)

                # Check if we've reached the end page
                if effective_end_page and page_number > effective_end_page:
                    if verbose:
                        print(f"Reached end page {effective_end_page}")
                    break

                # Add a delay between requests
                time.sleep(2)
                continue

            # Reset empty pages counter when we get results
            empty_pages = 0

            page_reviews_count = len(reviews)
            total_reviews += page_reviews_count

            if verbose:
                print(f"Found {page_reviews_count} reviews on page {page_number}")

            for review in reviews:
                try:
                    # Handle possible missing fields with .get()
                    data = {
                        'Page': review.get('_page_number', page_number),
                        'Date': pd.to_datetime(review.get("dates", {}).get("publishedDate")).strftime("%Y-%m-%d")
                               if review.get("dates", {}).get("publishedDate") else None,
                        'Author': review.get("consumer", {}).get("displayName", "Anonymous"),
                        'Body': review.get("text", ""),
                        'Heading': review.get("title", ""),
                        'Rating': review.get("rating"),
                        'Location': review.get("consumer", {}).get("countryCode", "")
                    }
                    reviews_data.append(data)
                except Exception as e:
                    if verbose:
                        print(f"Error processing a review: {e}")

            # Update progress info
            pbar.update(1)
            pages_scraped += 1
            pbar.set_postfix({"Reviews on page": page_reviews_count, "Total reviews": total_reviews})

            page_number += 1

            # Break if we've reached the end page
            if effective_end_page and page_number > effective_end_page:
                if verbose:
                    print(f"Reached end page {effective_end_page}")
                break

            # Add a delay between requests
            time.sleep(2)

    finally:
        # Always close the progress bar
        pbar.close()

    # Remove duplicates
    if reviews_data:
        original_count = len(reviews_data)
        df = pd.DataFrame(reviews_data)
        df.drop_duplicates(subset=['Body', 'Date', 'Author'], inplace=True)
        reviews_data = df.to_dict('records')
        duplicate_count = original_count - len(reviews_data)

        if verbose:
            print(f"\nScraping summary:")
            print(f"- Pages scraped: {pages_scraped}")
            print(f"- Page range: {start_page} to {page_number - 1}")
            print(f"- Raw reviews: {original_count}")
            print(f"- Duplicates removed: {duplicate_count}")
            print(f"- Final unique reviews: {len(reviews_data)}")

    return reviews_data

# Function to scrape in batches
def scrape_in_batches(base_url, batch_size=10, start_from=1, end_at=None):
    """
    Scrape Trustpilot reviews in batches of pages

    Parameters:
    - base_url: The base URL of the Trustpilot page
    - batch_size: Number of pages in each batch
    - start_from: First page to start scraping from
    - end_at: Last page to scrape (inclusive)
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    all_reviews = []

    batch_start = start_from
    batch_num = 1

    # Create a master progress bar for batch tracking
    batches_to_process = (end_at - start_from + 1) // batch_size + 1 if end_at else None

    if batches_to_process:
        master_pbar = tqdm(total=batches_to_process, desc="Processing batches", unit="batch")
    else:
        master_pbar = tqdm(desc="Processing batches", unit="batch")

    try:
        while True:
            batch_end = batch_start + batch_size - 1
            if end_at and batch_end > end_at:
                batch_end = end_at

            print(f"\n{'='*50}")
            print(f"PROCESSING BATCH #{batch_num}: Pages {batch_start}-{batch_end}")
            print(f"{'='*50}")

            batch_reviews = scrape_trustpilot_reviews(
                base_url,
                start_page=batch_start,
                end_page=batch_end
            )

            if batch_reviews:
                all_reviews.extend(batch_reviews)

                # Save this batch
                batch_df = pd.DataFrame(batch_reviews)
                batch_filename = f"{batch_folder}/firstport_reviews_batch{batch_num}_p{batch_start}-p{batch_end}_{timestamp}.csv"
                batch_df.to_csv(batch_filename, index=False)
                print(f"Saved batch #{batch_num} with {len(batch_reviews)} reviews to {batch_filename}")

                # Also save cumulative results
                all_df = pd.DataFrame(all_reviews)
                cumulative_filename = f"{batch_folder}/firstport_reviews_cumulative_{timestamp}.csv"
                all_df.to_csv(cumulative_filename, index=False)
                print(f"Updated cumulative results with {len(all_reviews)} total reviews")
            else:
                print(f"No reviews found in batch #{batch_num} (pages {batch_start}-{batch_end})")
                if batch_start > start_from:  # If not the first batch
                    print("Stopping batch processing as no more reviews were found")
                    break

            # Move to next batch
            batch_start = batch_end + 1
            batch_num += 1

            # Update master progress bar
            master_pbar.update(1)

            # Break if we've reached the end
            if end_at and batch_start > end_at:
                print(f"Reached specified end page ({end_at})")
                break

            print("\nPausing between batches (5 seconds)...")
            time.sleep(5)  # Longer pause between batches

    finally:
        # Always close the master progress bar
        master_pbar.close()

    return all_reviews

# Scrap

In [ ]:
# Create a folder for batch results
batch_folder = "firstport_reviews_batches"
os.makedirs(batch_folder, exist_ok=True)

In [ ]:

# Base URL for FirstPort on Trustpilot
BASE_URL = "https://www.trustpilot.com/review/www.firstport.co.uk"

# Example usage:
# 1. Scrape pages 1-10 as a single batch
# reviews = scrape_trustpilot_reviews(BASE_URL, start_page=1, end_page=10)

# 2. Scrape in batches of 20 pages, from page 1 to 100
reviews = scrape_in_batches(BASE_URL, batch_size=20, start_from=1, end_at=807)

# Create final DataFrame
df = pd.DataFrame(reviews)

Processing batches:   0%|          | 0/41 [00:00<?, ?batch/s]


PROCESSING BATCH #1: Pages 1-20
Scraping pages 1 to 20


Scraping pages: 0page [00:00, ?page/s]

Found 20 reviews on page 1
Found 20 reviews on page 2
Found 20 reviews on page 3
Found 20 reviews on page 4
Found 20 reviews on page 5
Found 20 reviews on page 6
Found 20 reviews on page 7
Found 20 reviews on page 8
Found 20 reviews on page 9
Found 20 reviews on page 10
Found 20 reviews on page 11
Found 20 reviews on page 12
Found 20 reviews on page 13
Found 20 reviews on page 14
Found 20 reviews on page 15
Found 20 reviews on page 16
Found 20 reviews on page 17
Found 20 reviews on page 18
Found 20 reviews on page 19
Found 20 reviews on page 20
Reached end page 20

Scraping summary:
- Pages scraped: 20
- Page range: 1 to 20
- Raw reviews: 400
- Duplicates removed: 0
- Final unique reviews: 400
Saved batch #1 with 400 reviews to firstport_reviews_batches/firstport_reviews_batch1_p1-p20_20250427_064739.csv
Updated cumulative results with 400 total reviews

Pausing between batches (5 seconds)...

PROCESSING BATCH #2: Pages 21-40
Scraping pages 21 to 40


Scraping pages: 0page [00:00, ?page/s]

...

Found 20 reviews on page 761
Found 20 reviews on page 762
Found 20 reviews on page 763
Found 20 reviews on page 764
Found 20 reviews on page 765
Found 20 reviews on page 766
Found 20 reviews on page 767
Found 20 reviews on page 768
Found 20 reviews on page 769
Found 20 reviews on page 770
Found 20 reviews on page 771
Found 20 reviews on page 772
Found 20 reviews on page 773
Found 20 reviews on page 774
Found 20 reviews on page 775
Found 20 reviews on page 776
Found 20 reviews on page 777
Found 20 reviews on page 778
Found 20 reviews on page 779
Found 20 reviews on page 780
Reached end page 780

Scraping summary:
- Pages scraped: 20
- Page range: 761 to 780
- Raw reviews: 400
- Duplicates removed: 0
- Final unique reviews: 400
Saved batch #39 with 400 reviews to firstport_reviews_batches/firstport_reviews_batch39_p761-p780_20250427_064739.csv
Updated cumulative results with 15600 total reviews

Pausing between batches (5 seconds)...

PROCESSING BATCH #40: Pages 781-800
Scraping pages 78

Scraping pages: 0page [00:00, ?page/s]

Found 20 reviews on page 781
Found 20 reviews on page 782
Found 20 reviews on page 783
Found 20 reviews on page 784
Found 20 reviews on page 785
Found 20 reviews on page 786
Found 20 reviews on page 787
Found 20 reviews on page 788
Found 20 reviews on page 789
Found 20 reviews on page 790
Found 20 reviews on page 791
Found 20 reviews on page 792
Found 20 reviews on page 793
Found 20 reviews on page 794
Found 20 reviews on page 795
Found 20 reviews on page 796
Found 20 reviews on page 797
Found 20 reviews on page 798
Found 20 reviews on page 799
Found 20 reviews on page 800
Reached end page 800

Scraping summary:
- Pages scraped: 20
- Page range: 781 to 800
- Raw reviews: 400
- Duplicates removed: 0
- Final unique reviews: 400
Saved batch #40 with 400 reviews to firstport_reviews_batches/firstport_reviews_batch40_p781-p800_20250427_064739.csv
Updated cumulative results with 16000 total reviews

Pausing between batches (5 seconds)...

PROCESSING BATCH #41: Pages 801-807
Scraping pages 80

Scraping pages: 0page [00:00, ?page/s]

Found 20 reviews on page 801
Found 20 reviews on page 802
Found 20 reviews on page 803
Found 20 reviews on page 804
Found 20 reviews on page 805
Found 20 reviews on page 806
Found 8 reviews on page 807
Reached end page 807

Scraping summary:
- Pages scraped: 7
- Page range: 801 to 807
- Raw reviews: 128
- Duplicates removed: 0
- Final unique reviews: 128
Saved batch #41 with 128 reviews to firstport_reviews_batches/firstport_reviews_batch41_p801-p807_20250427_064739.csv
Updated cumulative results with 16128 total reviews
Reached specified end page (807)


In [ ]:
# Display summary
print(f"\nFINAL RESULTS:")
print(f"Total reviews collected: {len(df)}")
if len(df) > 0:
    print("\nSample reviews:")
    display(df.head())

    # Number of reviews by rating
    print("\nDistribution by rating:")
    rating_counts = df['Rating'].value_counts().sort_index()
    display(rating_counts)

    # Save complete dataset to CSV
    filename = f"firstport_reviews_complete.csv"
    df.to_csv(filename, index=False)
    print(f"Saved all {len(df):,} reviews to {filename}")
else:
    print("No reviews collected, CSV not created.")


FINAL RESULTS:
Total reviews collected: 16000

Sample reviews:


,Page,Date,Author,Body,Heading,Rating,Location
0,1,2025-04-25,Douglas Armitt,Always gives 100%. Nothing is to much trouble ...,Always gives 100%,5,GB
1,1,2025-04-24,Catchy,Having moved into Chestnut Court 18 months ago...,Excellent site management.,5,GB
2,1,2025-04-24,Beth Dee,They have a bullying approach and do not liste...,They have a bullying approach and do…,1,GB
3,1,2025-04-24,stuart hurst,Donna Saunders has been a great help and is ve...,Donna Saunders,5,GB
4,1,2025-04-24,Stuart Gaylor,Been asking for statements for the RMC account...,Does anyone at FP care?,1,GB



Distribution by rating:


,count
Rating,
0,2
1,6580
2,206
3,162
4,662
5,8388


Saved all 16,000 reviews to firstport_reviews_complete.csv
